# 13 - every band, and pixel size WITHIN C-band (validation glacier SI; test only described)

**A100 GPU runtime**, standard RAM. Part 2 measured: 5.7 min. Nothing retrained, no result file changed.

1. **All bands** (CPU, ~1 min): pooled front error per glacier, band, satellite and pixel size,
   from the saved results, for the U-Net and the C-RADIO / satellite-DINOv3 probes and
   decoders. Descriptive: each group also differs in satellite, years and scenes.
2. **Pixel size within C-band** (GPU): the 21 SI RADARSAT images at 12 m are shrunk to 20 m
   and the saved models run on them. Same band and same satellite, so any change is pixel
   size alone. Control as in notebook 12 (heads must reproduce the saved results exactly).

**Pre-registered** (fixed on 2026-09-27 before the run; reproduced in docs/results.md): pixel size hurts a model if its
pooled error on the shrunk copies is at least 100 m worse than on the 12 m originals. The
r-rule of notebook 12 is reported too. L-band is not shrunk: 17 m to 20 m is too small a step.

In [1]:
# --- cell: mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# --- cell: secrets, GPU, RAM, data ---
import os, shutil
from pathlib import Path
import psutil, torch
from sartransfer.env import load_env, require, model_cache_on_drive
from sartransfer.data import inventory as inv

load_env()
require("HF_TOKEN", "gated model weights")
model_cache_on_drive()          # weights downloaded once, kept on Drive
assert torch.cuda.is_available(), "switch to a GPU runtime"
g = torch.cuda.get_device_properties(0)
assert "A100" in g.name, f"got {g.name}: use an A100 runtime -- the saved SI results were made on an A100, and the exact control needs the same GPU kernels"
print(f"gpu: {g.name} | GPU memory {g.total_memory / 1e9:.0f} GB | system RAM {psutil.virtual_memory().total / 1e9:.0f} GB "
      f"| CPU cores {os.cpu_count()}")
DRIVE = Path("/content/drive/MyDrive")
RAW, RES = DRIVE / "sar-transfer" / "raw", DRIVE / "sar-transfer" / "results"
inv.unzip_caffe(inv.fetch_caffe_zip(RAW), Path("/content/caffe"))
DATA = inv.find_data_root(Path("/content/caffe"))
df = inv.scan(DATA, subdirs=("sar_images",))
WORK = Path("/content/restest")
print(len(df), "SAR images | free disk:", f"{shutil.disk_usage('/content').free/1e9:.0f} GB")

In [4]:
# --- cell: PART 1 -- every band, pooled front error from the saved results (CPU, ~1 min) ---
import pandas as pd
from sartransfer.resolution import band_table, band_summary

bt = band_table(RES)
WORK.mkdir(parents=True, exist_ok=True)
bt.to_csv(WORK / "restest_bands_long.csv", index=False)
bs = band_summary(bt)
bs.to_csv(WORK / "restest_bands_summary.csv", index=False)
pd.set_option("display.width", 250, "display.max_rows", 200)
print("pooled MDE (m); decoders = mean of 3 seeds; satellite 'all' = pooled over satellites\n")
print(bs.round(0).to_string(index=False))

pooled MDE (m); decoders = mean of 3 seeds; satellite 'all' = pooled over satellites

split glacier band satellite  pixel_m  images  cradio_row1  cradio_row3  dinosat_row1  dinosat_row3   unet
   SI      SI    C      RSAT     12.0      21       1446.0       1264.0        1514.0        1121.0 1741.0
   SI      SI    C       all     12.0      21       1446.0       1264.0        1514.0        1121.0 1741.0
   SI      SI    C   ENVISAT     20.0      27       2002.0       1394.0        2049.0        1424.0 1222.0
   SI      SI    C       ERS     20.0      28       1831.0       1471.0        1582.0        1426.0 1110.0
   SI      SI    C      RSAT     20.0       6       3965.0       1140.0        3020.0        1562.0 1714.0
   SI      SI    C       all     20.0      61       2060.0       1403.0        1893.0        1434.0 1223.0
   SI      SI    L    PALSAR     17.0       7       1458.0       1544.0        3604.0        2530.0 2742.0
   SI      SI    L       all     17.0       7       1458.0

In [5]:
# --- cell: PART 2 -- C-band 12 m -> 20 m, same band and satellite (GPU, ~6 min) ---
from sartransfer.resolution import run_resolution_test

control, table = run_resolution_test(df, DATA, RES, WORK, token=os.environ["HF_TOKEN"], source=("C", 12.0, "RSAT"))
print(table[["model", "native", "shrunk_20m", "change_m", "pixel_size_effect", "C_20m", "r", "verdict",
             "median_change_per_image_m", "no_front_native", "no_front_20m"]]
      .round({"native": 0, "shrunk_20m": 0, "change_m": 0, "C_20m": 0, "r": 2, "median_change_per_image_m": 0})
      .to_string(index=False))

shrunk 21 SI RSAT images to 20 m pixels -> /content/restest/C12

--- 12 m (original, control): 21 C-band RSAT images ---
installing ['open_clip_torch']


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

remote code verified: 26 files == reviewed copy (nvidia/C-RADIOv4-H@0057b339059c)


Loading weights:   0%|          | 0/390 [00:01<?, ?it/s]

...

remote code verified: 26 files == reviewed copy (nvidia/C-RADIOv4-H@0057b339059c)


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

cradio-v4-h: 651.6 M parameters loaded
restricted to splits ('val',): 21 of 21 images
  21/21  0.0 min elapsed, 0.08 s/image, ETA 0.0 min
in memory: train X (0, 0) (0.0 GB fp16), 21 val images
extraction time: waiting for CPU preparation 0.0 min, encoder 0.0 min, keeping/copying 0.0 min, total 0.0 min


Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

dinov3-l-sat: 303.1 M parameters loaded
restricted to splits ('val',): 21 of 21 images
  21/21  0.0 min elapsed, 0.06 s/image, ETA 0.0 min
in memory: train X (0, 0) (0.0 GB fp16), 21 val images
extraction time: waiting for CPU preparation 0.0 min, encoder 0.0 min, keeping/copying 0.0 min, total 0.0 min

done in 5.7 min
          model  native  shrunk_20m  change_m                         pixel_size_effect  C_20m     r                                    verdict  median_change_per_image_m  no_front_native  no_front_20m
    cradio_row1  1446.0      1771.0     325.0 pixel size hurts (>= 100 m worse at 20 m) 2060.0  0.53          pixel size explains it (r >= 0.5)                      146.0                0             0
 cradio_row3_s0  1309.0      1523.0     214.0 pixel size hurts (>= 100 m worse at 20 m) 1497.0  1.14          pixel size explains it (r >= 0.5)                      126.0                0             0
 cradio_row3_s1  1330.0      1385.0      55.0                 no clear ch

In [6]:
# --- cell: save the tables to Drive (small CSVs) ---
dst = DRIVE / "sar-transfer" / "resolution_test"
dst.mkdir(parents=True, exist_ok=True)
for f in sorted(WORK.rglob("restest_*.csv")):
    shutil.copy2(f, dst / f.name)
print("saved:", sorted(p.name for p in dst.glob("restest_*.csv")))

saved: ['restest_C12_control.csv', 'restest_C12_images.csv', 'restest_C12_records_20m.csv', 'restest_C12_records_native.csv', 'restest_C12_table.csv', 'restest_bands_long.csv', 'restest_bands_summary.csv', 'restest_control.csv', 'restest_images.csv', 'restest_records_20m.csv', 'restest_records_7m.csv', 'restest_table.csv']
